# Musique Decomposer – Kaggle Notebook

**Goal:** Run the full decomposition pipeline on Musique test questions (2hop, 3hop, 4hop).

**Pipeline:**
1. Mask test questions with `dslim/bert-large-NER`
2. Embed masked pool + masked test queries with `intfloat/e5-small-v2`
3. Retrieve top-3 similar few-shot examples per question (cosine similarity)
4. Build prompts and run `Mistral-7B-Instruct-v0.3` (4-bit quantized)
5. Save decompositions + artifacts

In [ ]:
!pip install -q bitsandbytes sentence-transformers accelerate

In [ ]:
import json
import random
import sys
import hashlib
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import transformers

print(f"Python:       {sys.version}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"CUDA:         {torch.cuda.is_available()} / {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
if torch.cuda.is_available():
    print(f"VRAM:         {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ── Config ──
SEED = 42
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Kaggle dataset paths — adjust to match your uploaded dataset names
MASKED_POOL_PATH = Path("/kaggle/input/ner-pool/musique_few_shot_decompositions_ner_bert_large_NER.json")
DATA_REFINED_DIR = Path("/kaggle/input/ner-pool")

# Output
OUTPUT_DIR = Path("/kaggle/working/runs") / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Models
NER_MODEL = "dslim/bert-large-NER"
EMBED_MODEL = "intfloat/e5-small-v2"
LLM_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

# Experiment settings
GUIDED = True                 # include hop count in prompt
TOP_K = 3                     # number of few-shot examples to retrieve
SAMPLE_SIZE = None            # set to e.g. 50 for a quick test run, None for full
HOPS = [2, 3, 4]

# NER placeholder mapping
TYPE_TO_PLACEHOLDER = {
    "PER": "[PERSON]",
    "PERSON": "[PERSON]",
    "ORG": "[ORG]",
    "LOC": "[LOC]",
    "MISC": "[MISC]",
}

# Decoding
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.0
DO_SAMPLE = False

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

print(f"Run ID:       {RUN_ID}")
print(f"Device:       {DEVICE}")
print(f"Guided:       {GUIDED}")
print(f"Sample size:  {SAMPLE_SIZE or 'full'}")
print(f"Pool path:    {MASKED_POOL_PATH} (exists={MASKED_POOL_PATH.exists()})")
print(f"Data dir:     {DATA_REFINED_DIR} (exists={DATA_REFINED_DIR.exists()})")

In [ ]:
# ── Load data ──

# 1. Masked few-shot pool
pool_raw = json.loads(MASKED_POOL_PATH.read_text(encoding="utf-8"))
pool_items = []  # flat list of all pool items
for hop_key in sorted(pool_raw.keys()):
    for item in pool_raw[hop_key]:
        item["hop_key"] = hop_key
        pool_items.append(item)
print(f"Pool: {len(pool_items)} items ({', '.join(f'{k}: {len(v)}' for k, v in pool_raw.items())})")

# 2. Test questions (data_refined)
test_data = []  # list of {idx, question, decomposition, hop_count}
for h in HOPS:
    path = DATA_REFINED_DIR / f"musique_{h}hop_data_refined.json"
    if not path.exists():
        print(f"Warning: {path} not found, skipping {h}hop.")
        continue
    items = json.loads(path.read_text(encoding="utf-8"))
    for item in items:
        item["hop_count"] = h
    if SAMPLE_SIZE:
        items = random.sample(items, min(len(items), SAMPLE_SIZE))
    test_data.extend(items)
    print(f"  {h}hop: {len(items)} test questions")

print(f"\nTotal test questions: {len(test_data)}")

## Phase 1 — Mask test questions with NER

In [ ]:
# ── NER masking helpers ──

def apply_spans(text: str, spans: list[tuple[int, int, str]]) -> str:
    """Replace non-overlapping spans sorted by start position."""
    spans = sorted(spans, key=lambda x: x[0])
    merged = []
    last_end = -1
    for s, e, ph in spans:
        if s < last_end:
            continue
        merged.append((s, e, ph))
        last_end = e
    parts, last = [], 0
    for s, e, ph in merged:
        parts.append(text[last:s])
        parts.append(ph)
        last = e
    parts.append(text[last:])
    return "".join(parts)


def mask_questions_with_ner(questions: list[str], model_name: str, device: int) -> list[str]:
    """Mask all questions using HF token-classification pipeline."""
    from transformers import pipeline
    ner = pipeline(
        "token-classification",
        model=model_name,
        tokenizer=model_name,
        aggregation_strategy="max",
        device=device,
    )
    results = []
    for q in questions:
        entities = ner(q)
        spans = []
        for ent in entities:
            s, e = ent.get("start"), ent.get("end")
            if s is None or e is None:
                continue
            label = ent.get("entity_group") or ent.get("entity") or ent.get("label")
            ph = TYPE_TO_PLACEHOLDER.get(str(label), TYPE_TO_PLACEHOLDER.get(str(label).upper(), "[ENTITY]"))
            spans.append((int(s), int(e), ph))
        results.append(apply_spans(q, spans))
    del ner
    torch.cuda.empty_cache()
    return results

In [ ]:
# ── Run NER masking on test questions ──
print(f"Masking {len(test_data)} test questions with {NER_MODEL}...")
all_test_questions = [item["question"] for item in test_data]
ner_device = 0 if torch.cuda.is_available() else -1
masked_test_questions = mask_questions_with_ner(all_test_questions, NER_MODEL, ner_device)

masked_count = sum(1 for orig, m in zip(all_test_questions, masked_test_questions) if orig != m)
print(f"Done. {masked_count}/{len(all_test_questions)} questions had entities masked ({100*masked_count/len(all_test_questions):.1f}%).")

# Preview
for i in range(min(5, len(all_test_questions))):
    print(f"\n  Original: {all_test_questions[i]}")
    print(f"  Masked:   {masked_test_questions[i]}")

## Phase 2 — Embed pool + test queries

In [ ]:
# ── Embedding ──
from sentence_transformers import SentenceTransformer

def _needs_e5_prefix(model_id: str) -> bool:
    return "e5" in model_id.lower()

print(f"Loading embedding model: {EMBED_MODEL}")
embed_model = SentenceTransformer(EMBED_MODEL)
use_e5 = _needs_e5_prefix(EMBED_MODEL)

# Embed pool (masked texts)
pool_masked_texts = [item["masked"] for item in pool_items]
pool_to_encode = [f"passage: {t}" for t in pool_masked_texts] if use_e5 else pool_masked_texts
pool_embeddings = embed_model.encode(pool_to_encode, normalize_embeddings=True, show_progress_bar=True)
print(f"Pool embeddings: {pool_embeddings.shape}")

# Embed test queries (masked)
test_to_encode = [f"query: {t}" for t in masked_test_questions] if use_e5 else masked_test_questions
test_embeddings = embed_model.encode(test_to_encode, normalize_embeddings=True, show_progress_bar=True)
print(f"Test embeddings: {test_embeddings.shape}")

del embed_model
torch.cuda.empty_cache()
print("Embedding model freed.")

## Phase 3 — Retrieve few-shot examples

In [ ]:
# ── Similarity retrieval ──
# Cosine similarity = dot product (vectors are L2-normalized)
sim_matrix = np.dot(test_embeddings, pool_embeddings.T)  # (n_test, n_pool)

# For each test question, get top-k pool indices
few_shot_indices = np.argsort(-sim_matrix, axis=1)[:, :TOP_K]  # (n_test, TOP_K)

# Preview retrieval for first 3 test questions
for i in range(min(3, len(test_data))):
    print(f"\nTest Q{i+1} ({test_data[i]['hop_count']}hop): {all_test_questions[i]}")
    print(f"  Masked: {masked_test_questions[i]}")
    for rank, pool_idx in enumerate(few_shot_indices[i]):
        score = sim_matrix[i, pool_idx]
        pi = pool_items[pool_idx]
        print(f"  #{rank+1} (sim={score:.4f}) [{pi['hop_key']}] {pi['masked']}")

## Phase 4 — Decomposer (Mistral 7B, 4-bit)

In [ ]:
# ── Load LLM ──
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {LLM_MODEL} (4-bit)...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=False,
)
model.eval()
print("Model loaded.")
if torch.cuda.is_available():
    mem = torch.cuda.memory_allocated() / 1e9
    print(f"GPU memory used: {mem:.1f} GB")

In [ ]:
# ── Prompt template + helpers ──

PROMPT_GUIDED = """You decompose questions into atomic steps.

Rules:
- The number of steps MUST equal the hop count.
- Output ONLY the steps.
- Do NOT explain.
- Do NOT repeat the question.
- Each step must be a single factual question.
- Use [#1], [#2], etc. ONLY to refer to previous step results.
- One step per line.

Examples:

{few_shot_examples}

Task:
Hop count: {hop_count}
Question: {question}
Decomposition:
"""

PROMPT_UNGUIDED = """You decompose questions into atomic steps.

Rules:
- Decompose into the minimal number of atomic steps.
- Output ONLY the steps.
- Do NOT explain.
- Do NOT repeat the question.
- Each step must be a single factual question.
- Use [#1], [#2], etc. ONLY to refer to previous step results.
- One step per line.

Examples:

{few_shot_examples}

Task:
Question: {question}
Decomposition:
"""


def format_few_shot_examples(examples: list[dict], hop_count: int | None = None) -> str:
    """Format retrieved pool items into prompt examples."""
    blocks = []
    for ex in examples:
        if hop_count is not None:
            block = f"Hop count: {hop_count}\nQuestion: {ex['question']}\nDecomposition:\n{ex['decomposition']}"
        else:
            block = f"Question: {ex['question']}\nDecomposition:\n{ex['decomposition']}"
        blocks.append(block)
    return "\n\n".join(blocks)


def build_prompt(question: str, hop_count: int, few_shot_str: str) -> str:
    if GUIDED:
        return PROMPT_GUIDED.format(
            question=question, hop_count=hop_count, few_shot_examples=few_shot_str,
        )
    return PROMPT_UNGUIDED.format(question=question, few_shot_examples=few_shot_str)


def generate_decomposition(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()
    return response.split("Question:")[0].strip()

In [ ]:
# ── Run inference ──
results = []
prompts_dir = OUTPUT_DIR / "prompts_log"
prompts_dir.mkdir(exist_ok=True)

print(f"Running decomposer on {len(test_data)} questions...")
for i, item in enumerate(test_data):
    q = item["question"]
    hop = item["hop_count"]

    # Retrieve few-shot examples
    fs_items = [pool_items[idx] for idx in few_shot_indices[i]]
    fs_scores = [float(sim_matrix[i, idx]) for idx in few_shot_indices[i]]
    hop_input = hop if GUIDED else None
    few_shot_str = format_few_shot_examples(fs_items, hop_input)

    # Build prompt and generate
    prompt = build_prompt(q, hop, few_shot_str)
    decomposition = generate_decomposition(prompt)

    results.append({
        "idx": item.get("idx"),
        "question": q,
        "hop_count": hop,
        "gold_decomposition": item.get("decomposition"),
        "predicted_decomposition": decomposition,
        "masked_question": masked_test_questions[i],
        "few_shot_scores": fs_scores,
    })

    # Log every 10th prompt
    if (i + 1) % 10 == 0:
        log_path = prompts_dir / f"prompt_{i+1:04d}_{hop}hop.txt"
        header = (
            f"Question (original): {q}\n"
            f"Question (masked):   {masked_test_questions[i]}\n"
            f"Top {TOP_K} similar:\n"
        )
        for rank, (fi, fs) in enumerate(zip(fs_items, fs_scores), 1):
            header += f"  {rank}. sim={fs:.4f} | {fi['masked']}\n"
        log_path.write_text(
            header + "\n--- Prompt ---\n" + prompt + "\n--- Response ---\n" + decomposition,
            encoding="utf-8",
        )

    if (i + 1) % 25 == 0 or (i + 1) == len(test_data):
        print(f"  [{i+1}/{len(test_data)}] {hop}hop | {q[:60]}...")
        print(f"    → {decomposition[:120]}")

print(f"\nInference complete. {len(results)} decompositions generated.")

## Evaluation — Gold vs Predicted Decomposition

**Metrics:**
- **Step count accuracy** — does the model output the correct number of steps?
- **Exact match (EM)** — normalized full-chain match
- **Token F1 (order-aware)** — step *i* predicted vs step *i* gold, bag-of-words F1
- **Token F1 (order-unaware)** — best bipartite matching between predicted and gold steps
- **ROUGE-L (step-level)** — longest common subsequence between predicted and gold steps

All metrics computed per-hop and overall.

In [ ]:
# ── Evaluation helpers ──
import re
from collections import Counter


def normalize(text: str) -> str:
    """Lowercase, strip punctuation and extra whitespace."""
    text = text.lower().strip()
    text = re.sub(r"[^\w\s#\[\]]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def parse_steps(text) -> list[str]:
    """Parse a decomposition into individual steps.
    Handles: list of strings, numbered lines ('1. ...'), or plain newline-separated."""
    if isinstance(text, list):
        return [str(s).strip() for s in text if str(s).strip()]
    if not isinstance(text, str):
        return []
    lines = [l.strip() for l in text.strip().split("\n") if l.strip()]
    steps = []
    for line in lines:
        cleaned = re.sub(r"^\d+[\.\)\-]\s*", "", line).strip()
        if cleaned:
            steps.append(cleaned)
    return steps


def token_f1(pred: str, gold: str) -> tuple[float, float, float]:
    """Bag-of-words token-level precision, recall, F1 between two strings."""
    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()
    if not pred_tokens and not gold_tokens:
        return 1.0, 1.0, 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0, 0.0, 0.0
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_common = sum(common.values())
    if num_common == 0:
        return 0.0, 0.0, 0.0
    precision = num_common / len(pred_tokens)
    recall = num_common / len(gold_tokens)
    f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1


def lcs_length(a: list, b: list) -> int:
    """Length of longest common subsequence."""
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[m][n]


def rouge_l(pred: str, gold: str) -> float:
    """ROUGE-L F1 between two strings (token-level LCS)."""
    pred_tokens = normalize(pred).split()
    gold_tokens = normalize(gold).split()
    if not pred_tokens and not gold_tokens:
        return 1.0
    if not pred_tokens or not gold_tokens:
        return 0.0
    lcs = lcs_length(pred_tokens, gold_tokens)
    if lcs == 0:
        return 0.0
    prec = lcs / len(pred_tokens)
    rec = lcs / len(gold_tokens)
    return 2 * prec * rec / (prec + rec)


def evaluate_decomposition(pred_text, gold_text) -> dict:
    """Compute all metrics for a single (predicted, gold) pair."""
    pred_steps = parse_steps(pred_text)
    gold_steps = parse_steps(gold_text)

    n_pred, n_gold = len(pred_steps), len(gold_steps)
    step_count_correct = int(n_pred == n_gold)

    # Exact match (normalized full chain)
    pred_norm = " ".join(normalize(s) for s in pred_steps)
    gold_norm = " ".join(normalize(s) for s in gold_steps)
    exact_match = int(pred_norm == gold_norm)

    # Order-aware token F1: align step i to step i
    ordered_f1s = []
    for i in range(max(n_pred, n_gold)):
        p = pred_steps[i] if i < n_pred else ""
        g = gold_steps[i] if i < n_gold else ""
        _, _, f1 = token_f1(p, g)
        ordered_f1s.append(f1)
    ordered_f1 = np.mean(ordered_f1s) if ordered_f1s else 0.0

    # Order-unaware token F1: best matching (greedy)
    unordered_f1s = []
    remaining_gold = list(range(n_gold))
    for i in range(n_pred):
        best_f1, best_j = 0.0, -1
        for j in remaining_gold:
            _, _, f1 = token_f1(pred_steps[i], gold_steps[j])
            if f1 > best_f1:
                best_f1, best_j = f1, j
        unordered_f1s.append(best_f1)
        if best_j >= 0:
            remaining_gold.remove(best_j)
    # Penalize missing gold steps
    for _ in remaining_gold:
        unordered_f1s.append(0.0)
    unordered_f1 = np.mean(unordered_f1s) if unordered_f1s else 0.0

    # ROUGE-L per step (order-aware)
    rouge_scores = []
    for i in range(max(n_pred, n_gold)):
        p = pred_steps[i] if i < n_pred else ""
        g = gold_steps[i] if i < n_gold else ""
        rouge_scores.append(rouge_l(p, g))
    avg_rouge_l = np.mean(rouge_scores) if rouge_scores else 0.0

    return {
        "n_pred_steps": n_pred,
        "n_gold_steps": n_gold,
        "step_count_correct": step_count_correct,
        "exact_match": exact_match,
        "ordered_token_f1": float(ordered_f1),
        "unordered_token_f1": float(unordered_f1),
        "rouge_l": float(avg_rouge_l),
    }


print("Evaluation functions ready.")

In [ ]:
# ── Run evaluation ──
print("Evaluating decompositions against gold...\n")

eval_results = []
for r in results:
    ev = evaluate_decomposition(r["predicted_decomposition"], r["gold_decomposition"])
    ev["hop_count"] = r["hop_count"]
    eval_results.append(ev)

# ── Per-hop and overall metrics ──
def aggregate_metrics(evals: list[dict]) -> dict:
    if not evals:
        return {}
    return {
        "n": len(evals),
        "step_count_acc": np.mean([e["step_count_correct"] for e in evals]),
        "exact_match": np.mean([e["exact_match"] for e in evals]),
        "ordered_token_f1": np.mean([e["ordered_token_f1"] for e in evals]),
        "unordered_token_f1": np.mean([e["unordered_token_f1"] for e in evals]),
        "rouge_l": np.mean([e["rouge_l"] for e in evals]),
    }

overall = aggregate_metrics(eval_results)
per_hop = {}
for h in HOPS:
    hop_evals = [e for e in eval_results if e["hop_count"] == h]
    if hop_evals:
        per_hop[h] = aggregate_metrics(hop_evals)

# ── Print table ──
header = f"{'Hop':<8} {'N':>5} {'StepAcc':>9} {'EM':>8} {'Ord-F1':>9} {'Unord-F1':>10} {'ROUGE-L':>9}"
print(header)
print("─" * len(header))
for h in HOPS:
    if h not in per_hop:
        continue
    m = per_hop[h]
    print(f"{h}hop{'':<4} {m['n']:>5} {m['step_count_acc']:>9.3f} {m['exact_match']:>8.3f} {m['ordered_token_f1']:>9.3f} {m['unordered_token_f1']:>10.3f} {m['rouge_l']:>9.3f}")
print("─" * len(header))
print(f"{'ALL':<8} {overall['n']:>5} {overall['step_count_acc']:>9.3f} {overall['exact_match']:>8.3f} {overall['ordered_token_f1']:>9.3f} {overall['unordered_token_f1']:>10.3f} {overall['rouge_l']:>9.3f}")

# Attach eval to each result for saving
for r, ev in zip(results, eval_results):
    r["eval"] = ev

## Save artifacts

In [ ]:
# ── Save results ──
(OUTPUT_DIR / "results.json").write_text(
    json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8"
)

# Config snapshot
config = {
    "seed": SEED,
    "run_id": RUN_ID,
    "device": DEVICE,
    "guided": GUIDED,
    "top_k": TOP_K,
    "sample_size": SAMPLE_SIZE,
    "hops": HOPS,
    "ner_model": NER_MODEL,
    "embed_model": EMBED_MODEL,
    "llm_model": LLM_MODEL,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "do_sample": DO_SAMPLE,
    "pool_path": str(MASKED_POOL_PATH),
    "data_dir": str(DATA_REFINED_DIR),
    "total_test_questions": len(test_data),
    "total_pool_items": len(pool_items),
}
(OUTPUT_DIR / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

# Metrics (including evaluation)
hop_counts = {}
for r in results:
    h = r["hop_count"]
    hop_counts[h] = hop_counts.get(h, 0) + 1

metrics = {
    "total_questions": len(results),
    "questions_per_hop": {str(k): v for k, v in hop_counts.items()},
    "avg_few_shot_sim": float(np.mean([r["few_shot_scores"][0] for r in results])) if results else 0,
    "evaluation": {
        "overall": {k: float(v) if isinstance(v, (float, np.floating)) else v for k, v in overall.items()},
        "per_hop": {str(h): {k: float(v) if isinstance(v, (float, np.floating)) else v for k, v in m.items()} for h, m in per_hop.items()},
    },
}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

# Run notes
notes = f"""# Decomposer Run — {RUN_ID}
- Model: {LLM_MODEL} (4-bit)
- Guided: {GUIDED}
- NER masking: {NER_MODEL}
- Embedding: {EMBED_MODEL}
- Pool: {len(pool_items)} items
- Test questions: {len(results)}
- Per hop: {hop_counts}
- Avg top-1 similarity: {metrics['avg_few_shot_sim']:.4f}

## Evaluation
- Step count accuracy: {overall['step_count_acc']:.3f}
- Exact match: {overall['exact_match']:.3f}
- Ordered token F1: {overall['ordered_token_f1']:.3f}
- Unordered token F1: {overall['unordered_token_f1']:.3f}
- ROUGE-L: {overall['rouge_l']:.3f}
"""
(OUTPUT_DIR / "notes.md").write_text(notes, encoding="utf-8")

print(f"Artifacts saved to: {OUTPUT_DIR}")
print(f"  results.json  ({len(results)} items)")
print(f"  config.json")
print(f"  metrics.json")
print(f"  notes.md")
print(f"  prompts_log/  ({len(list(prompts_dir.glob('*.txt')))} logged)")

In [ ]:
# ── Quick preview of results ──
for h in HOPS:
    hop_results = [r for r in results if r["hop_count"] == h]
    if not hop_results:
        continue
    print(f"\n{'='*80}")
    print(f"  {h}-hop examples (first 3)")
    print(f"{'='*80}")
    for r in hop_results[:3]:
        gold = r["gold_decomposition"]
        if isinstance(gold, list):
            gold = "\n".join(f"  {j+1}. {s}" for j, s in enumerate(gold))
        else:
            gold = "  " + str(gold).replace("\n", "\n  ")
        print(f"\nQ: {r['question']}")
        print(f"Gold:\n{gold}")
        print(f"Predicted:\n  {r['predicted_decomposition']}")